In [ ]:
import pathlib

import matplotlib.pyplot as plt

import brain_probability_flux.analyzer.hierarchical_clustering as hc
import brain_probability_flux.plotting as bpp

In [ ]:
# Results are read from data/. To plot a fresh run, set DATA_DIR to its run
# directory, for example Path("../data/2026-10-20").
DATA_DIR = bpp.DATA_DIR

In [ ]:
max_fig_width = bpp.FIG_WIDTH
max_fig_height = bpp.FIG_HEIGHT
panel_gap = bpp.PANEL_GAP_PT / 72  # gap between panels in the composite (in)

CLUSTER_SIZES = list(range(4, 7))  # K=4,5,6 — K=7 shown in main text Fig. 3a
N_TASKS = 8
out_base = pathlib.Path("../figures/fig-s07")

## Observed flux on the hypercube edges for K = 4, 5, 6 clusters

The tree is cut into exactly K clusters. The state space has 2^K states.

In [ ]:
flux_flat_per_k = []
for K in CLUSTER_SIZES:
    pfa, _ = bpp.aggregated_pfa(f"maxclust_{K}", DATA_DIR)
    flux_flat_per_k.append(bpp.observed_edge_flux(pfa)[0])

## Task cosine-similarity matrices (a-c) and dendrograms (d-f)

In [ ]:
# Full task names — no abbreviation
task_names = bpp.TASK_NAMES_FULL

# Each column also holds about 14 pt of tick labels beyond `panel_size`.
panel_size = (max_fig_width - 2 * panel_gap) / len(CLUSTER_SIZES) - 14 / 72
# The clustermaps (a-c) are drawn smaller than `panel_size` because their
# tick labels and title add about 48 pt to the cropped panel.
panel_size_row0 = 1.62  # in
Z_tasks_per_k = []

for idx, (K, flux_flat) in enumerate(zip(CLUSTER_SIZES, flux_flat_per_k)):
    panel_label = chr(ord("a") + idx)

    g, ax, Z_task = hc.plot_clustermap(
        flux_flat.T,
        matrix_type="cosine-similarity",
        figsize=(panel_size_row0, panel_size_row0),
        return_linkage_matrices=True,
        ax_label=None,
        cbar_label="Cos. sim.",
    )
    Z_tasks_per_k.append(Z_task)

    # Task name tick labels in dendrogram leaf order (full names)
    col_order = g.dendrogram_col.reordered_ind
    row_order = g.dendrogram_row.reordered_ind
    x_lbl = [task_names[i] for i in col_order]
    y_lbl = [task_names[i] for i in row_order]

    n = N_TASKS
    ax.set_xticks([i + 0.5 for i in range(n)])
    ax.set_xticklabels(x_lbl, rotation=90)
    ax.xaxis.set_tick_params(length=0)
    ax.set_yticks([i + 0.5 for i in range(n)])
    ax.set_yticklabels(y_lbl, rotation=0, ha="left")
    ax.yaxis.set_tick_params(length=0)

    # Panel b (idx == 1): flip y-axis so rows read bottom-to-top,
    # matching the orientation of the other panels.
    if idx == 1:
        ax.invert_yaxis()
        g.ax_row_dendrogram.invert_yaxis()
        ax.invert_xaxis()
        g.ax_col_dendrogram.invert_xaxis()
    if idx == 2:
        ax.invert_xaxis()
        ax.invert_yaxis()
        g.ax_col_dendrogram.invert_xaxis()
        g.ax_row_dendrogram.invert_yaxis()

    g.ax_col_dendrogram.set_title(f"{K} clusters")

    # Cropped to the content: the title and tick labels added above extend
    # past the clustermap's canvas.
    bpp.save_panel(g.figure, out_base, panel_label, bbox_inches="tight")
    plt.close(g.figure)

# ── Dendrogram panels (d, e, f) ──────────────────────────────────────────────
# Same method as fig-03b: plot_dendrogram on the task linkage matrix.
# Generate all three first to find a shared y-limit before saving.
dendro_h = panel_size  # square panels so dendrograms are not squished
dendro_figs: list = []
dendro_axes: list = []

for idx, (K, Z_task) in enumerate(zip(CLUSTER_SIZES, Z_tasks_per_k)):
    fig_dn, ax_dn, _ = hc.plot_dendrogram(
        Z_task,
        threshold=0,
        return_dn=True,
        figsize=(panel_size, dendro_h),
        labels=task_names,
        leaf_font_size=7,
        draw_threshold=False,
    )
    # Flip x-axis for panels e and f to match orientation of panels b and c above
    if idx in (1, 2):
        ax_dn.invert_xaxis()
    dendro_figs.append(fig_dn)
    dendro_axes.append(ax_dn)

# Shared y-limit: use the maximum across all three dendrograms
ylim_max = max(ax.get_ylim()[1] for ax in dendro_axes)

for idx, (fig_dn, ax_dn) in enumerate(zip(dendro_figs, dendro_axes)):
    ax_dn.set_ylim(0, ylim_max)
    panel_label = chr(ord("d") + idx)
    bpp.save_panel(fig_dn, out_base, panel_label, bbox_inches="tight")
    plt.close(fig_dn)